In [1]:
from pathlib import Path
from io import StringIO

import pandas as pd

current_path = Path.cwd()
project_root = current_path if (current_path / "requirements.txt").exists() else current_path.parent
data_dir = project_root / "data"
input_csv_path = data_dir / "sample_cleaned.csv"

print("pandas_version:", pd.__version__)
print("project_root:", project_root)
print("data_dir_exists:", data_dir.exists())
print("input_csv_path:", input_csv_path)
print("input_csv_exists:", input_csv_path.exists())

pandas_version: 2.0.3
project_root: /Users/im-youngchan/Desktop/US Financial
data_dir_exists: True
input_csv_path: /Users/im-youngchan/Desktop/US Financial/data/sample_cleaned.csv
input_csv_exists: True


In [11]:
if input_csv_path.exists():
    previous_sample_df = pd.read_csv(input_csv_path)
    print("previous_sample_row_count:", len(previous_sample_df))
    print("previous_sample_column_count:", len(previous_sample_df.columns))
    display(previous_sample_df.head())
else:
    print("previous sample CSV is not available in this environment.")

previous_sample_row_count: 3
previous_sample_column_count: 9


,company_name,ticker,sector,revenue_usd_m,operating_income_usd_m,net_income_usd_m,unit,frequency,source_note
0,Apple Inc.,APPL,consumer technology,1000,300,250,USD million,annual_sample,class_sample
1,Microsoft Corporation,MSFT,software and cloud,900,320,260,USD million,annual_sample,class_sample
2,NVIDIA Corporation,NVDA,semiconductor,700,280,220,USD million,annual_sample,class_sample


In [13]:
date_sample_csv_text = """company_name,ticker,sector,report_date,revenue_usd_m_text,operating_income_usd_m_text,net_income_usd_m_text,unit,frequency,source_note
Apple Inc.,AAPL,consumer technology,2024-12-31,"1,000",300,250,USD million,annual_sample,class_sample
Microsoft Corporation,MSFT,software and cloud,2024-12-31,900,320,260,USD million,annual_sample,class_sample
NVIDIA Corporation,NVDA,semiconductor,2024-12-31,,280,220,USD million,annual_sample,class_sample
Apple Inc.,AAPL,consumer technology,2023-12-31,950,290,230,USD million,annual_sample,class_sample
Microsoft Corporation,MSFT,software and cloud,2023-12-31,880,,240,USD million,annual_sample,class_sample
NVIDIA Corporation,NVDA,semiconductor,2023-12-31,650,220,not available,USD million,annual_sample,class_sample
"""

raw_date_df = pd.read_csv(StringIO(date_sample_csv_text))

print("raw_row_count:", len(raw_date_df))
print("raw_column_count:", len(raw_date_df.columns))
raw_date_df

raw_row_count: 6
raw_column_count: 10


,company_name,ticker,sector,report_date,revenue_usd_m_text,operating_income_usd_m_text,net_income_usd_m_text,unit,frequency,source_note
0,Apple Inc.,AAPL,consumer technology,2024-12-31,"1,000",300.0,250,USD million,annual_sample,class_sample
1,Microsoft Corporation,MSFT,software and cloud,2024-12-31,900,320.0,260,USD million,annual_sample,class_sample
2,NVIDIA Corporation,NVDA,semiconductor,2024-12-31,NaN,280.0,220,USD million,annual_sample,class_sample
3,Apple Inc.,AAPL,consumer technology,2023-12-31,950,290.0,230,USD million,annual_sample,class_sample
4,Microsoft Corporation,MSFT,software and cloud,2023-12-31,880,NaN,240,USD million,annual_sample,class_sample
5,NVIDIA Corporation,NVDA,semiconductor,2023-12-31,650,220.0,not available,USD million,annual_sample,class_sample


In [7]:
raw_date_df["report_date"] = pd.to_datetime(raw_date_df["report_date"])

print("date_dtype:", raw_date_df["report_date"].dtype)
print("min_report_date:", raw_date_df["report_date"].min().date())
print("max_report_date:", raw_date_df["report_date"].max().date())
raw_date_df[["ticker", "report_date"]]

date_dtype: datetime64[ns]
min_report_date: 2023-12-31
max_report_date: 2024-12-31


,ticker,report_date
0,AAPL,2024-12-31
1,MSFT,2024-12-31
2,NVDA,2024-12-31
3,AAPL,2023-12-31
4,MSFT,2023-12-31
5,NVDA,2023-12-31


In [8]:
numeric_source_columns = [
    "revenue_usd_m_text",
    "operating_income_usd_m_text",
    "net_income_usd_m_text",
]

numeric_clean_columns = [
    "revenue_usd_m",
    "operating_income_usd_m",
    "net_income_usd_m",
]

for source_column, clean_column in zip(numeric_source_columns, numeric_clean_columns):
    raw_date_df[clean_column] = pd.to_numeric(
        raw_date_df[source_column].astype("string").str.replace(",", "", regex=False),
        errors="coerce",
    )

raw_date_df[
    [
        "ticker",
        "report_date",
        "revenue_usd_m_text",
        "revenue_usd_m",
        "operating_income_usd_m_text",
        "operating_income_usd_m",
        "net_income_usd_m_text",
        "net_income_usd_m",
    ]
]

,ticker,report_date,revenue_usd_m_text,revenue_usd_m,operating_income_usd_m_text,operating_income_usd_m,net_income_usd_m_text,net_income_usd_m
0,AAPL,2024-12-31,"1,000",1000,300.0,300.0,250,250
1,MSFT,2024-12-31,900,900,320.0,320.0,260,260
2,NVDA,2024-12-31,NaN,<NA>,280.0,280.0,220,220
3,AAPL,2023-12-31,950,950,290.0,290.0,230,230
4,MSFT,2023-12-31,880,880,NaN,<NA>,240,240
5,NVDA,2023-12-31,650,650,220.0,220.0,not available,<NA>


In [9]:
clean_columns = [
    "company_name",
    "ticker",
    "sector",
    "report_date",
    "revenue_usd_m",
    "operating_income_usd_m",
    "net_income_usd_m",
    "unit",
    "frequency",
    "source_note",
]

date_cleaned_df = raw_date_df[clean_columns].copy()

print("date_cleaned_row_count:", len(date_cleaned_df))
print("date_cleaned_column_count:", len(date_cleaned_df.columns))
date_cleaned_df

date_cleaned_row_count: 6
date_cleaned_column_count: 10


,company_name,ticker,sector,report_date,revenue_usd_m,operating_income_usd_m,net_income_usd_m,unit,frequency,source_note
0,Apple Inc.,AAPL,consumer technology,2024-12-31,1000,300.0,250,USD million,annual_sample,class_sample
1,Microsoft Corporation,MSFT,software and cloud,2024-12-31,900,320.0,260,USD million,annual_sample,class_sample
2,NVIDIA Corporation,NVDA,semiconductor,2024-12-31,<NA>,280.0,220,USD million,annual_sample,class_sample
3,Apple Inc.,AAPL,consumer technology,2023-12-31,950,290.0,230,USD million,annual_sample,class_sample
4,Microsoft Corporation,MSFT,software and cloud,2023-12-31,880,<NA>,240,USD million,annual_sample,class_sample
5,NVIDIA Corporation,NVDA,semiconductor,2023-12-31,650,220.0,<NA>,USD million,annual_sample,class_sample


In [10]:
date_sorted_df = date_cleaned_df.sort_values(["report_date", "ticker"]).copy()

print("first_report_date:", date_sorted_df["report_date"].iloc[0].date())
print("last_report_date:", date_sorted_df["report_date"].iloc[-1].date())
date_sorted_df[["report_date", "ticker", "revenue_usd_m", "net_income_usd_m"]]

first_report_date: 2023-12-31
last_report_date: 2024-12-31


,report_date,ticker,revenue_usd_m,net_income_usd_m
3,2023-12-31,AAPL,950,230
4,2023-12-31,MSFT,880,240
5,2023-12-31,NVDA,650,<NA>
0,2024-12-31,AAPL,1000,250
1,2024-12-31,MSFT,900,260
2,2024-12-31,NVDA,<NA>,220
